# To Run Spectral Ratio Illumination Demo on Google Colab

Omar Elmady 

Wednesday, Dec 11 

CS 7180 

## Setup Steps:
1. **Enable GPU Runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Update model link** in Step 2 if you have it from your professor
3. **Run all cells in order** (Runtime → Run all)

## What this notebook does:
- Checks GPU availability
- Downloads model from Google Drive (if link provided)
- Clones your GitHub repository
- Installs all dependencies (PyTorch with GPU/CPU support)
- Runs validation checks
- Processes images with your algorithms
- Downloads results as a .tar.gz file

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi || echo "No GPU detected - will use CPU (slower but works)"

## Step 2: Clone Repository and Download Model

### 2a. Clone Repository

First, clone the GitHub repository to get all the code and data.

In [ ]:
import os

# Remove if already exists from previous runs
if os.path.exists('/content/Spectral_Ratio_Illumination_Demo'):
    !rm -rf /content/Spectral_Ratio_Illumination_Demo

# Clone the repository
print("📥 Cloning repository from GitHub...")
!git clone https://github.com/oelmady/Spectral_Ratio_Illumination_Demo.git /content/Spectral_Ratio_Illumination_Demo

# Change to the project directory
%cd /content/Spectral_Ratio_Illumination_Demo

print("\nRepository cloned successfully!")
print("\n📁 Repository contents:")
!ls -lh

### 2b. Download Model from Google Drive

To extract file ID from a Drive link like `https://drive.google.com/file/d/1ABC123XYZ/view?usp=sharing`, copy the `1ABC123XYZ` part. If you need to change it, update `MODEL_DRIVE_ID` below.  

The model will download directly into the repository's `model/` folder.

In [ ]:
import os

# ============================================
# CONFIGURATION: Google Drive file ID for model
# ============================================
MODEL_DRIVE_ID = "1h2fVtLQJpgLl4_C3MLA_VDuqlJTcAqf6"
# ============================================

if MODEL_DRIVE_ID:
    print("📥 Downloading model from Google Drive...")
    
    # Install gdown for Drive downloads
    !pip install -q gdown
    
    import gdown
    
    # Download directly into the repo's model/ directory
    model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'
    url = f'https://drive.google.com/uc?id={MODEL_DRIVE_ID}'
    
    try:
        gdown.download(url, model_path, quiet=False)
        
        # Verify download
        if os.path.exists(model_path):
            size_mb = os.path.getsize(model_path) / (1024 * 1024)
            if size_mb > 100:  # Should be ~528MB
                print(f"\nModel downloaded successfully: {size_mb:.1f} MB")
                print(f"   Location: {model_path}")
            else:
                print(f"\nModel file seems too small ({size_mb:.1f} MB)")
                print("   Check if the Drive link allows public access")
        else:
            print("\n❌ Model download failed")
            print("   Make sure the file is shared with 'Anyone with the link'")
    except Exception as e:
        print(f"\n❌ Error downloading model: {e}")
        print("   Double-check the file ID and sharing permissions")
else:
    print("No model file ID provided")
    print("   Will run baseline-only experiments (no neural ISD prediction)")

# Show model directory contents
print("\n📁 Model directory:")
!ls -lh /content/Spectral_Ratio_Illumination_Demo/model/


## Step 3: Install Dependencies

This cell installs all required Python packages:
- PyTorch (GPU version if available, otherwise CPU)
- OpenCV (full version with GUI support)
- NumPy, Matplotlib, and other dependencies

In [ ]:
import subprocess
import sys

print("Installing dependencies...\n")

# Upgrade pip
print("1️⃣ Upgrading pip...")
!pip install --quiet --upgrade pip setuptools wheel

# Install opencv and scientific computing libraries
print("\n2️⃣ Installing OpenCV, NumPy, Matplotlib, scikit-image...")
!pip install --quiet opencv-python numpy matplotlib scikit-image scipy

# Install PyTorch with GPU support if available
print("\n3️⃣ Installing PyTorch...")
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("   GPU detected: Installing CUDA-enabled PyTorch (cu118)")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
except:
    print("   No GPU: Installing CPU-only PyTorch")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install any remaining requirements
print("\n4️⃣ Installing remaining requirements...")
!pip install --quiet -r requirements.txt 2>/dev/null || true

print("\nAll dependencies installed successfully!")

# Verify installations
print("\n📋 Checking installed versions:")
import torch
import cv2
import numpy as np
from skimage import __version__ as skimage_version
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   scikit-image: {skimage_version}")

## Step 4: Run Preflight Check

This validates that everything is set up correctly.

In [ ]:
%cd /content/Spectral_Ratio_Illumination_Demo
!python preflight_check.py

## Step 5: Run Experiments

culatioThis cell processes all images in `data/images/` with four algorithms.
- Neural ISD prediction
- SR-constrained Retinex
- Baseline Retinex (for comparison)
- SR-based color correction

In [ ]:
import os
import sys

%cd /content/Spectral_Ratio_Illumination_Demo

# Set PYTHONPATH environment variable so subprocess can find modules
os.environ['PYTHONPATH'] = '/content/Spectral_Ratio_Illumination_Demo'

model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:  # > 1MB
    print("🚀 Running FULL experiment (with neural ISD model)...")
    print("   This includes: model inference + SR-Retinex + baseline Retinex + color correction")
    print("   Plus: Quality metrics (SSIM + color constancy)")
    print("   Using optimized parameters: iterations=3, sigma=25, distance=1.0\n")
    !python scripts/run_batch.py \
        --use-model \
        --retinex \
        --baseline-retinex \
        --sr-correct \
        --gray-world \
        --white-patch \
        --multiscale-retinex \
        --compute-metrics \
        --iterations 5 \
        --sigma 25 \
        --distance 1.0
else:
    print("Running BASELINE experiment (no model)...")
    print("   This includes: baseline Retinex + SR-constrained Retinex (using annotated maps)\n")
    print("   Note: Since no model is available, this will use pre-existing SR maps")
    print("   from data/sr_maps/ if available, or skip SR-constrained processing.\n")
    !python scripts/run_batch.py \
        --baseline-retinex \
        --retinex \
        --compute-metrics \
        --iterations 3 \
        --sigma 25

print("\nProcessing complete! Check results/ directory for images and quality_metrics.json")

## Step 6: View Sample Results

Display a few output images to verify processing worked correctly.

## View Quality Metrics

Display the quantitative comparison between your SR-constrained method and the baseline.

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

metrics_file = 'results/quality_metrics.json'

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    print("📊 Quality Metrics Summary\n")
    print("="*70)
    
    # Collect data per method
    sr_color_errors = []
    sr_ssims = []
    baseline_color_errors = []
    baseline_ssims = []
    sr_corr_color_errors = []
    sr_corr_ssims = []
    
    for img_name, img_metrics in metrics.items():
        if 'sr_retinex' in img_metrics:
            sr_color_errors.append(img_metrics['sr_retinex']['color_constancy_error_deg'])
            sr_ssims.append(img_metrics['sr_retinex']['ssim'])
        if 'baseline_retinex' in img_metrics:
            baseline_color_errors.append(img_metrics['baseline_retinex']['color_constancy_error_deg'])
            baseline_ssims.append(img_metrics['baseline_retinex']['ssim'])
        if 'sr_color_correction' in img_metrics:
            sr_corr_color_errors.append(img_metrics['sr_color_correction']['color_constancy_error_deg'])
            sr_corr_ssims.append(img_metrics['sr_color_correction']['ssim'])
    
    # Print table
    print(f"{'Method':<30} {'Avg Color Error (°)':<20} {'Avg SSIM':<15}")
    print("-"*70)
    
    if sr_color_errors:
        print(f"{'SR-Constrained Retinex':<30} {np.mean(sr_color_errors):<20.2f} {np.mean(sr_ssims):<15.4f}")
    if baseline_color_errors:
        print(f"{'Baseline Retinex':<30} {np.mean(baseline_color_errors):<20.2f} {np.mean(baseline_ssims):<15.4f}")
    if sr_corr_color_errors:
        print(f"{'SR Color Correction':<30} {np.mean(sr_corr_color_errors):<20.2f} {np.mean(sr_corr_ssims):<15.4f}")
    
    print("="*70)
    
    # Visualize comparison
    if sr_color_errors and baseline_color_errors:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Color error comparison
        methods = ['SR-Constrained', 'Baseline', 'SR Color Corr']
        color_means = [np.mean(sr_color_errors), np.mean(baseline_color_errors), 
                      np.mean(sr_corr_color_errors) if sr_corr_color_errors else 0]
        colors = ['#2ecc71', '#e74c3c', '#3498db']
        
        axes[0].bar(methods, color_means, color=colors, alpha=0.7, edgecolor='black')
        axes[0].set_ylabel('Color Constancy Error (degrees)', fontsize=12)
        axes[0].set_title('Color Preservation\n(Lower is Better)', fontsize=14, fontweight='bold')
        axes[0].grid(axis='y', alpha=0.3)
        
        # SSIM comparison
        ssim_means = [np.mean(sr_ssims), np.mean(baseline_ssims),
                     np.mean(sr_corr_ssims) if sr_corr_ssims else 0]
        
        axes[1].bar(methods, ssim_means, color=colors, alpha=0.7, edgecolor='black')
        axes[1].set_ylabel('SSIM Score', fontsize=12)
        axes[1].set_title('Structural Similarity\n(Higher is Better)', fontsize=14, fontweight='bold')
        axes[1].set_ylim([0, 1])
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Calculate improvement
        if sr_color_errors and baseline_color_errors:
            improvement = np.mean(baseline_color_errors) - np.mean(sr_color_errors)
            improvement_pct = (improvement / np.mean(baseline_color_errors)) * 100
            
            print(f"\nYour SR-constrained method shows {improvement:.2f}° less color error!")
            print(f"   That's a {improvement_pct:.1f}% improvement in color preservation!")
else:
    print("No metrics file found. Make sure Step 5 completed with --compute-metrics flag.")

## Step 7: Package and Download Results

This creates a `.tar.gz` archive of all results and downloads it to your local machine.

In [ ]:
import os
from google.colab import files

%cd /content/Spectral_Ratio_Illumination_Demo

# Determine what to package
has_results = os.path.exists('results')
has_tuning = os.path.exists('results_tuning')

if has_tuning:
    # If tuning was run, package all tuning results
    print("Packaging parameter tuning results...")
    !tar -czf results_tuning.tar.gz results_tuning/ 2>/dev/null
    
    if os.path.exists('results_tuning.tar.gz'):
        size_mb = os.path.getsize('results_tuning.tar.gz') / (1024 * 1024)
        print(f"Archive created: results_tuning.tar.gz ({size_mb:.1f} MB)")
        print("\nDownloading to your computer...")
        files.download('results_tuning.tar.gz')
        print("Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results_tuning.tar.gz")
    else:
        print("Failed to create tuning archive")

elif has_results:
    # If only single run results exist, package those
    print("Packaging results...")
    !tar -czf results.tar.gz results/ 2>/dev/null
    
    if os.path.exists('results.tar.gz'):
        size_mb = os.path.getsize('results.tar.gz') / (1024 * 1024)
        print(f"Archive created: results.tar.gz ({size_mb:.1f} MB)")
        print("\nDownloading to your computer...")
        files.download('results.tar.gz')
        print("Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results.tar.gz")
    else:
        print("Failed to create results archive")

else:
    print("No results to package. Make sure Step 5 or parameter tuning completed successfully.")

## View Sample Results (Step 5 outputs)

This displays results from **Step 5** (the main experiment run) for presentation.

**Shows 4 method comparison:**
- Original input (8-bit reference)
- Baseline Retinex (standard method - may shift colors)
- SR-constrained Retinex (new method - preserves colors better)
- SR color correction (simple illumination adjustment)

**Expected Results:**
- SR-constrained should preserve color better than baseline (less color shifts)
- Baseline may over-smooth or change colors unrealistically
- SR correction should brighten shadows while maintaining color relationships

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

results_dir = 'results'

if not os.path.exists(results_dir):
    print("No results found. Run Step 5 first.")
else:
    # Find all result files
    all_files = sorted(os.listdir(results_dir))
    
    # Find a base image name (extract from 8-bit reference images)
    base_names = set()
    for f in all_files:
        if f.endswith('_image_8bit.png'):
            base_names.add(f.replace('_image_8bit.png', ''))
    
    if not base_names:
        print("No processed images found in results/")
    else:
        # Pick first image for visualization
        base_name = sorted(base_names)[0]
        print(f"📸 Showing method comparison for: {base_name}\n")
        
        # Define the comparison sequence with correct file naming
        comparisons = [
            (f'{base_name}_image_8bit.png', 'Original Input (8-bit)'),
            (f'{base_name}_baseline_retinex_vis.png', 'Baseline Retinex'),
            (f'{base_name}_sr_retinex_vis.png', 'SR-Constrained Retinex\n'),
            (f'{base_name}_sr_shifted_vis.png', 'SR Color Correction')
        ]
        
        # Load images
        images_to_show = []
        labels_to_show = []
        
        for filename, label in comparisons:
            img_path = os.path.join(results_dir, filename)
            if os.path.exists(img_path):
                images_to_show.append(img_path)
                labels_to_show.append(label)
            else:
                print(f"   Missing: {filename}")
        
        if len(images_to_show) >= 2:
            # Display in 2x2 grid
            n_images = len(images_to_show)
            fig, axes = plt.subplots(2, 2, figsize=(16, 16))
            axes = axes.flatten()
            
            for idx, (img_path, label) in enumerate(zip(images_to_show, labels_to_show)):
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].set_title(label, fontsize=16, fontweight='bold', pad=10)
                axes[idx].axis('off')
            
            # Hide unused subplots
            for idx in range(n_images, 4):
                axes[idx].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"\nComparison displayed for: {base_name}")
            print("\nWhat to look for in your presentation:")
            print("   1. Color preservation: SR-constrained should maintain colors better than baseline")
            print("   2. Detail recovery: Check shadow regions - should reveal detail without artifacts")
            print("   3. Natural appearance: SR methods should look more realistic than baseline")
            print("   4. Brightness: All corrected images should be brighter but with preserved colors")
            
            # Show additional images if available
            if len(base_names) > 1:
                print(f"\n{len(base_names)} images processed. Showing first one.")
                print(f"   Other images: {', '.join(sorted(base_names)[1:3])}")
        else:
            print("Not enough output images found. Make sure Step 5 completed successfully.")
            print(f"   Found files: {images_to_show}")